# Lección 11 · ML + optimización: de la vida útil al plan de mantenimiento

**Qué vas a aprender**
- **Predict-then-optimize**: un modelo de ML predice, un MILP decide. Cómo se conectan sin perder la incertidumbre.
- Predecir la **vida útil remanente** (RUL) de las multiplicadoras de un parque eólico con *gradient boosting*.
- **Fuga de datos** (*leakage*): por qué validar por fila miente, y cómo validar por turbina (`GroupKFold`).
- El error del modelo **no es parejo**: crece con la distancia a la falla.
- **Cuantiles** y **calibración**: un RUL pesimista que de verdad cubra el 90 %.
- Tres formas de pasar la predicción al MILP: el punto, el cuantil y los **escenarios** (la lección 8).
- Por qué la métrica que importa es el **costo de la decisión**, no el MAE.
- El modelo crece: de 40 a 640 turbinas. Escribir cada modelo en **notación matemática** y leerla símbolo por símbolo.

**Cómo usar este notebook**
- Corre las celdas en orden con `Shift + Enter`.
- Antes de cada celda, adivina qué va a imprimir.

## 0. Preparar el entorno
Hoy entra **scikit-learn**, la librería de ML de Python. Colab ya la trae.

In [ ]:
%pip install -q pyomo highspy scikit-learn

In [ ]:
import itertools
import logging
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.repn import generate_standard_repn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, KFold

SOLVER = pyo.SolverFactory("appsi_highs")
logging.getLogger("pyomo.contrib.appsi").setLevel(logging.ERROR)


def eur(x):
    return f"{x:,.0f}".replace(",", ".") + " €"


def tamano(m):
    # El tamaño del modelo: variables, restricciones y no ceros (coeficientes distintos de 0)
    variables = list(m.component_data_objects(pyo.Var))
    restricciones = list(m.component_data_objects(pyo.Constraint, active=True))
    return {"variables": len(variables),
            "binarias": sum(v.is_binary() for v in variables),
            "restricciones": len(restricciones),
            "no_ceros": sum(sum(c != 0 for c in generate_standard_repn(r.body).linear_coefs) for r in restricciones)}

## 1. Dónde quedamos
En la lección 8 la incertidumbre venía **dada**: "invierno duro con 15 %". Alguien te entregó los escenarios.
Hoy los escenarios salen **de los datos**: un modelo de ML predice cuándo falla cada equipo, y sus errores te dicen cuánto confiar.

**Analogía.** El GPS y el reporte de tráfico. El MILP es el GPS: da la mejor ruta con las calles que conoce.
El ML es el reporte de tráfico: predice cómo estarán las calles. Si el GPS cree a ciegas que la Aleja Trzech Wieszczów estará libre, te mete al trancón.
Un buen GPS dice "hay 30 % de que se tranque" y elige la ruta que aguanta.

**El caso.** Un parque eólico en el Báltico con turbinas que tienen una **multiplicadora** (la caja de engranajes entre el rotor y el generador).
Tiene sensores: temperatura y presión del aceite, vibración. Cuando se daña, la temperatura y la vibración suben y la presión baja.
- Si la cuadrilla la cambia **antes** de que falle, se pierde la vida que le quedaba: **1.000 € por día** de vida desperdiciada.
- Si **falla** antes del cambio: grúa de urgencia, turbina parada semanas. **60.000 €**.
- Hay **una cuadrilla**: cambia una multiplicadora por día.

## 2. Tres turbinas, una cuadrilla
El modelo de ML (sección 3) predice cuántos días le quedan a cada multiplicadora. Pero se equivoca.
Con sus errores pasados armamos 3 escenarios igual de probables por turbina:

| Turbina | RUL predicho (días) | Escenarios de RUL real |
|---|---|---|
| T1 | 3 | 1, 3, 5 |
| T2 | 4 | 2, 4, 6 |
| T3 | 6 | 3, 6, 9 |

El horizonte es de 6 días. Cada turbina se mantiene un día del 1 al 6, o **no se mantiene** en el horizonte (día 0).

**Las 3 piezas:**
- **Decido:** qué día cambio la multiplicadora de cada turbina (o si no la cambio en estos 6 días).
- **Objetivo:** minimizar el costo esperado: vida desperdiciada + fallas.
- **Restricciones:** cada turbina se asigna a un día (o a "no"); la cuadrilla hace una por día.

El costo de cambiar el día $d$ si el RUL real es $r$:
- $d \le r$: llega a tiempo. Desperdicia $r - d$ días: $1.000 \cdot (r - d)$ €.
- $d > r$: llegó tarde. Falla: 60.000 €.
- No se cambia ($d = 0$): falla si $r \le 6$ (dentro del horizonte). Si $r > 6$, el problema queda para el próximo plan: 0 €.

In [ ]:
VALOR_DIA, FALLA = 1_000, 60_000


def costo(d, rul, H):
    # d: día del cambio (0 = no se cambia en el horizonte). rul: uno o varios RUL reales. Devuelve un costo por RUL.
    rul = np.asarray(rul, dtype=float)
    if d == 0:
        return np.where(rul <= H, FALLA, 0.0)
    return np.where(d <= rul, VALOR_DIA * (rul - d), FALLA)


CHICO = pd.DataFrame({"turbina": ["T1", "T2", "T3"], "rul_pred": [3, 4, 6], "escenarios": [[1, 3, 5], [2, 4, 6], [3, 6, 9]]})
H_CHICO = 6
print("T1 cambiada el día 3, si el RUL real es 1, 3 o 5:", costo(3, [1, 3, 5], H_CHICO))
CHICO

### 2.1 Fuerza bruta: probar todos los planes
Cada turbina elige un día de 0 a 6: hay $7^3 = 343$ planes. Quitamos los que ponen dos turbinas el mismo día.
Cada plan tiene dos costos:
- el que **cree** el plan si el RUL predicho fuera exacto,
- el **esperado** con los 3 escenarios de cada turbina.

In [ ]:
filas = []
for plan in itertools.product(range(H_CHICO + 1), repeat=len(CHICO)):
    dias = [d for d in plan if d > 0]
    if len(dias) != len(set(dias)):  # la cuadrilla hace una por día
        continue
    cree = sum(costo(d, r, H_CHICO).item() for d, r in zip(plan, CHICO["rul_pred"]))
    esperado = sum(costo(d, esc, H_CHICO).mean() for d, esc in zip(plan, CHICO["escenarios"]))
    filas.append({"plan (T1, T2, T3)": plan, "cree": cree, "esperado": round(esperado)})
planes_chico = pd.DataFrame(filas)
print("Planes factibles:", len(planes_chico), "de", 7 ** 3)
pd.concat([planes_chico.nsmallest(1, "cree"), planes_chico.nsmallest(3, "esperado")])

**Lectura:**
- Si crees el pronóstico, el mejor plan es **(3, 4, 6)**: cada turbina el último día de su RUL predicho. Cree que cuesta **0 €**.
  Su costo esperado es **62.333 €**: en el escenario bajo de cada turbina, llega tarde.
- El mejor plan con los escenarios es **(1, 2, 3)**: cada turbina el último día de su escenario **más bajo**. Cuesta **7.000 €** esperados.
  Paga unos días de vida desperdiciada para casi no arriesgar fallas.
- El pronóstico era el mismo. Cambió **cómo** se lo pasaste al optimizador.

**Analogía.** Tu hijo tiene que estar en el *przedszkole* a las 8:00 y el tranvía tarda "20 minutos". Si sales a las 7:40 en punto, una de cada tres mañanas llegas tarde.
Sales a las 7:30: pierdes 10 minutos casi siempre, pero nunca llegas tarde.

### En notación
Conjuntos: $E$ = turbinas, $D = \{0, 1, \dots, H\}$ = días (0 = no se cambia), $S_e$ = escenarios de RUL de la turbina $e$.
Datos: $r_{es}$ = RUL de la turbina $e$ en el escenario $s$, $K$ = cuadrillas.
Variable: $x_{ed} \in \{0, 1\}$ = 1 si la turbina $e$ se cambia el día $d$.

$$
\begin{aligned}
\min \quad & \sum_{e \in E} \sum_{d \in D} \bar{c}_{ed}\, x_{ed} \\
\text{s.a.} \quad & \sum_{d \in D} x_{ed} = 1 && \forall\, e \in E && \text{(un día por turbina)} \\
& \sum_{e \in E} x_{ed} \le K && \forall\, d \in D \setminus \{0\} && \text{(cuadrillas)} \\
& x_{ed} \in \{0, 1\}
\end{aligned}
\qquad \text{con} \qquad
\bar{c}_{ed} = \frac{1}{\lvert S_e \rvert} \sum_{s \in S_e} c(d, r_{es})
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\hat{r}_e$ | "erre gorro de e" | el RUL **predicho** de la turbina $e$ | `rul_pred` |
| $r_{es}$ | "erre de e en ese" | el RUL en el escenario $s$ | `escenarios` |
| $c(d, r)$ | "costo de cambiar el día d si el RUL es r" | desperdicio o falla | `costo(d, r, H)` |
| $\bar{c}_{ed}$ | "ce barra" | costo **esperado**: el promedio sobre los escenarios | `costo(d, esc, H).mean()` |
| $D \setminus \{0\}$ | "D menos el cero" | los días de trabajo; el día 0 no gasta cuadrilla | `range(1, H + 1)` |

**Fíjate:** en la lección 8 cada escenario tenía su copia de variables ($e_s$, $x_{bfs}$), porque la segunda etapa **decidía** algo (desde qué base enviar).
Aquí la segunda etapa no decide nada: si llegas tarde, falla, y ya. Entonces el valor esperado se **precalcula** en $\bar{c}_{ed}$ (como el clima en la lección 7)
y el modelo estocástico tiene **el mismo tamaño** que el determinista.

In [ ]:
def tabla_costos(rul_por_turbina, H):
    # rul_por_turbina: {turbina: lista de RUL posibles}. Devuelve la tabla larga (turbina, dia, costo esperado).
    filas = [(e, d, costo(d, r, H).mean()) for e, r in rul_por_turbina.items() for d in range(H + 1)]
    return pd.DataFrame(filas, columns=["turbina", "dia", "costo"])


def plan_mantenimiento(costos, cuadrillas=1):
    claves = list(zip(costos["turbina"], costos["dia"]))
    c = dict(zip(claves, costos["costo"]))
    E = list(costos["turbina"].unique())
    dias = sorted(d for d in costos["dia"].unique() if d > 0)
    m = pyo.ConcreteModel()
    m.K = pyo.Set(initialize=claves, dimen=2)
    m.x = pyo.Var(m.K, within=pyo.Binary)
    m.obj = pyo.Objective(expr=sum(c[k] * m.x[k] for k in m.K), sense=pyo.minimize)
    m.un_dia = pyo.Constraint(E, rule=lambda m, e: sum(m.x[e, d] for d in range(len(dias) + 1)) == 1)
    m.cuadrilla = pyo.Constraint(dias, rule=lambda m, d: sum(m.x[e, d] for e in E) <= cuadrillas)
    inicio = time.perf_counter()
    SOLVER.solve(m)
    m.segundos = time.perf_counter() - inicio
    m.dia = {e: next(d for d in range(len(dias) + 1) if pyo.value(m.x[e, d]) > 0.5) for e in E}
    return m


m_punto = plan_mantenimiento(tabla_costos(dict(zip(CHICO["turbina"], CHICO["rul_pred"].map(lambda r: [r]))), H_CHICO))
m_esc = plan_mantenimiento(tabla_costos(dict(zip(CHICO["turbina"], CHICO["escenarios"])), H_CHICO))
print("Con el punto:      ", m_punto.dia, "| cree", eur(pyo.value(m_punto.obj)))
print("Con los escenarios:", m_esc.dia, "| esperado", eur(pyo.value(m_esc.obj)))
print("Tamaño:", tamano(m_esc))

**Lectura:**
- El MILP llega a los mismos dos planes de la fuerza bruta: **(3, 4, 6)** con el punto y **(1, 2, 3)** con los escenarios, 7.000 €.
- 3 turbinas × 7 días = 21 binarias, 3 + 6 = 9 restricciones.

**Los dos tamaños.**
- Modelo: $\lvert E \rvert (H + 1)$ binarias y $\lvert E \rvert + H$ restricciones. **Lineal** en turbinas y en días. Los escenarios **no** agregan variables: solo cambian $\bar{c}_{ed}$.
- Espacio de búsqueda: $(H + 1)^{\lvert E \rvert}$. Con 40 turbinas y 60 días, $61^{40} \approx 10^{71}$.
- La estructura es la **asignación** de la lección 4 (turbinas a días, con cupo $K$ por día): la matriz es TU y el LP ya da enteros.

## 3. El modelo de ML: predecir la vida útil
### 3.1 Los datos
Tenemos el historial de **100 multiplicadoras** que ya fallaron (*run-to-failure*). Una fila por turbina y día, con sus sensores.
Como no tenemos los datos reales, los **simulamos** (semilla fija), al estilo del dataset C-MAPSS de la NASA:
- Cada multiplicadora vive entre 120 y 300 días y se daña más rápido o más lento.
- Cada una tiene **su propio nivel base** en los sensores (no todas las máquinas salen iguales de fábrica).
- El daño crece cuadrático con la edad: al final, la temperatura y la vibración suben rápido.

**Features** de ventana: promedio y cambio de los últimos 10 días de cada sensor. Solo miran **hacia atrás**: el día $t$ no usa nada del día $t + 1$.

In [ ]:
SENSORES = ["temp_aceite", "presion_aceite", "vibracion"]


def simular(n, semilla, censurar=False):
    # censurar=True: la turbina sigue en operación; vemos sus datos hasta un día al azar antes de la falla
    rng = np.random.default_rng(semilla)
    tablas = []
    for e in range(n):
        vida, ritmo = int(rng.integers(120, 300)), rng.uniform(0.6, 1.4)
        base_t, base_p, base_v = rng.uniform(52, 68), rng.uniform(2.7, 3.3), rng.uniform(0.12, 0.28)
        fin = int(rng.integers(40, vida - 5)) if censurar else vida
        t = np.arange(fin)
        dano = (t / vida) ** 2 * ritmo
        tablas.append(pd.DataFrame({"turbina": f"T{e:03d}", "dia": t, "rul": vida - t,
                                    "temp_aceite": base_t + 25 * dano + rng.normal(0, 1.5, fin),
                                    "presion_aceite": base_p - 0.8 * dano + rng.normal(0, 0.06, fin),
                                    "vibracion": base_v + 1.5 * dano + rng.normal(0, 0.08, fin)}))
    df = pd.concat(tablas, ignore_index=True)
    por_turbina = df.groupby("turbina")
    for s in SENSORES:
        df[f"{s}_m10"] = por_turbina[s].transform(lambda x: x.rolling(10, min_periods=1).mean())  # promedio 10 días
        df[f"{s}_d10"] = por_turbina[s].transform(lambda x: x.diff(10)).fillna(0)  # cambio en 10 días
    return df


historia = simular(100, semilla=23)
X_COLS = [c for c in historia.columns if c not in ("turbina", "rul")]
print("Filas:", len(historia), "| turbinas:", historia["turbina"].nunique(), "| features:", len(X_COLS))
historia.head()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.2))
for eje, s in zip(ejes, SENSORES):
    for e, g in list(historia.groupby("turbina"))[:6]:
        eje.plot(g["rul"], g[f"{s}_m10"], lw=1)
    eje.set(xlabel="días hasta la falla (RUL)", title=s)
    eje.invert_xaxis()
plt.suptitle("6 multiplicadoras: cada una con su nivel base; todas se disparan al final")
plt.tight_layout()
plt.show()

**Lectura:**
- Lejos de la falla (RUL de 200), los sensores están planos: casi no dicen nada. Cerca (RUL < 50), se disparan.
- Cada curva arranca en un nivel distinto (mira la presión). Ese nivel base no dice nada del daño, pero sirve para **reconocer** a cada turbina. Guarda esa idea para la sección 3.2.

**El objetivo $y$:** el RUL, **recortado a 150 días**. El plan mira 60 días: que el modelo distinga 200 de 250 no sirve para nada y le quita esfuerzo donde importa.

In [ ]:
X = historia[X_COLS]
y = historia["rul"].clip(upper=150)
grupos = historia["turbina"]
print("RUL recortado:", y.describe().round(1).to_dict())

### 3.2 La fuga de datos: validar por fila contra validar por turbina
Para medir el error hay que predecir datos que el modelo **no vio**: validación cruzada con 5 partes (*5-fold*).

- **Por fila** (`KFold`): mezcla las filas al azar. El día 101 de la T007 queda en test, y los días 100 y 102 de la T007 quedan en train.
- **Por turbina** (`GroupKFold`): cada turbina queda **entera** en train o entera en test. Es lo que pasa en la vida real: el modelo predice turbinas que nunca vio.

**Analogía.** Un examen de polaco con las mismas frases que estudiaste, cambiando una palabra. Sacas 5, pero no sabes polaco.

**Las métricas.** MAE (*mean absolute error*): en promedio, ¿por cuántos días se equivoca?

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
$$

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $y_i$, $\hat{y}_i$ | "ye", "ye gorro" | valor real y valor predicho de la fila $i$ | `y`, `modelo.predict(X)` |
| $\lvert \cdot \rvert$ | "valor absoluto" | distancia sin signo | `abs()` |
| $\text{MAE}$ | "em a e" | error medio, en días | `mean_absolute_error(y, pred)` |
| $g(i)$ | "grupo de i" | la turbina de la fila $i$: todas sus filas van juntas | `groups=grupos` |

In [ ]:
def gb(**kw):
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=0, **kw)


def validar(fabrica, partir):
    # Predicción fuera de la muestra (out-of-fold): cada fila la predice un modelo que no la vio
    oof = np.zeros(len(y))
    for train, test in partir.split(X, y, grupos if isinstance(partir, GroupKFold) else None):
        oof[test] = fabrica().fit(X.iloc[train], y.iloc[train]).predict(X.iloc[test])
    return oof


inicio = time.perf_counter()
oof_fila = validar(gb, KFold(5, shuffle=True, random_state=0))
oof = validar(gb, GroupKFold(5))
print(f"MAE validando por fila:    {mean_absolute_error(y, oof_fila):.1f} días")
print(f"MAE validando por turbina: {mean_absolute_error(y, oof):.1f} días")
print(f"({time.perf_counter() - inicio:.0f} s)")

**Lectura:**
- Por fila, el modelo parece equivocarse **10 días**. Por turbina, **15 días**. La validación por fila promete un error un tercio más bajo del real.
- Por qué: con filas vecinas de la misma turbina en train, el modelo **reconoce la turbina** por su nivel base y su edad. No aprende a predecir la falla: memoriza.
- Con turbinas nuevas, esa trampa no existe. **El número que se reporta es el de `GroupKFold`.**

Otras fugas típicas: features que usan el futuro (un promedio centrado en vez de hacia atrás), normalizar con la media de todo el dataset antes de partir,
o en series de tiempo, validar con datos anteriores a los de train.

### 3.3 ¿Es bueno 15 días? Contra qué compararlo
Un número solo no dice nada. Se compara con modelos simples (*baselines*).

In [ ]:
oof_lineal = validar(LinearRegression, GroupKFold(5))
comparacion = pd.DataFrame({"modelo": ["promedio de y (sin features)", "regresión lineal", "gradient boosting"],
                            "MAE_dias": [mean_absolute_error(y, np.full(len(y), y.mean())),
                                         mean_absolute_error(y, oof_lineal), mean_absolute_error(y, oof)]}).round(1)
comparacion

**Lectura:**
- Sin features, el error es de **44 días**. La regresión lineal baja a **19,5**. *Gradient boosting* a **15**.
- El *boosting* gana porque el daño es **no lineal**: los sensores están planos y de pronto se disparan. Un árbol corta en "vibración > umbral"; una recta no.

### 3.4 El error no es parejo
El MAE promedia todas las filas. Pero al plan le importa sobre todo lo que pasa **cerca** de la falla. Miremos el error por tramo del RUL predicho.

In [ ]:
errores = pd.DataFrame({"pred": oof, "real": y, "error": y - oof})
errores["tramo"] = pd.cut(errores["pred"], [-1, 20, 40, 60, 90, 151], labels=["0-20", "20-40", "40-60", "60-90", "90-150"])
por_tramo = errores.groupby("tramo", observed=True)["error"].agg(
    filas="count", MAE=lambda e: e.abs().mean(), sesgo="mean", p10=lambda e: e.quantile(0.10), p90=lambda e: e.quantile(0.90)).round(1)
por_tramo

**Lectura:**
- Cuando el modelo dice "le quedan menos de 20 días", se equivoca por **7 días** en promedio. Cuando dice "40 a 90", por **17 a 19**.
- La columna `p10`: en el 10 % de los casos, el RUL real es **al menos** tantos días **menor** que el predicho. Con predicción de 40 a 60 días, 27 días menos.
- El error crece con la distancia a la falla: lejos, los sensores todavía no muestran el daño.
- Esta tabla es la materia prima de los **escenarios**: para una turbina con predicción $\hat{r}$, sus escenarios son $\hat{r}$ + los errores de su tramo.

### 3.5 Un RUL pesimista: el cuantil 10 %
En vez de predecir el valor esperado, se puede entrenar el modelo para predecir el **cuantil 10 %**: un valor que el RUL real supera el 90 % de las veces.
Se cambia la función de pérdida por la **pérdida pinball** (*quantile loss*):

$$
\rho_\tau(y - \hat{y}) = \begin{cases} \tau\, (y - \hat{y}) & \text{si } y \ge \hat{y} \\ (1 - \tau)\, (\hat{y} - y) & \text{si } y < \hat{y} \end{cases}
$$

Con $\tau = 0{,}1$, pasarse (predecir más de lo real) cuesta 9 veces más que quedarse corto. El modelo aprende a ser prudente.

| Símbolo | Se lee | Significa | En Python |
|---|---|---|---|
| $\tau$ | "tau" | el cuantil que se quiere: 0,1 = el 10 % de abajo | `quantile=0.10` |
| $\rho_\tau$ | "ro tau" | pérdida pinball: castiga distinto pasarse y quedarse corto | `loss="quantile"` |
| $P(y \ge \hat{q})$ | "probabilidad de que y supere a q gorro" | la **cobertura**: ideal 90 % | `(y >= q10).mean()` |

In [ ]:
oof_q10 = validar(lambda: gb(loss="quantile", quantile=0.10), GroupKFold(5))
cerca = oof <= 60  # turbinas que el modelo ve dentro del horizonte de 60 días
print(f"Cobertura del cuantil 10 % (ideal 90 %): todas las filas {(y >= oof_q10).mean():.0%} | predicción <= 60 días {(y[cerca] >= oof_q10[cerca]).mean():.0%}")

**Lectura:**
- El cuantil "10 %" cubre el **83 %** de las filas, no el 90 %. Donde importa (predicción ≤ 60 días), el **81 %**: 1 de cada 5 turbinas falla antes de su RUL "pesimista", el doble de lo prometido.
- Un modelo de cuantil **no garantiza** su cobertura: aprende con turbinas de train, y las de test son otras.

### 3.6 Calibración: correr el cuantil hasta que cubra
**Calibración conformal** (*conformalized quantile regression*): con las predicciones fuera de la muestra, mide cuánto falla el cuantil y córrelo esa distancia.

$$
\delta = \text{cuantil}_{0{,}9}\big( \hat{q}_i - y_i \big), \qquad \hat{q}^{\,\text{cal}} = \hat{q} - \delta
$$

Si en el 90 % de los casos el cuantil se pasa por menos de $\delta$ días, restarle $\delta$ lo deja cubriendo el 90 %. Se mide en la zona que importa: predicción ≤ 60 días.

In [ ]:
delta = np.quantile((oof_q10 - y)[cerca], 0.90)
q10_cal = oof_q10 - delta
print(f"delta = {delta:.1f} días")
print(f"Cobertura calibrada: predicción <= 60 días {(y[cerca] >= q10_cal[cerca]).mean():.0%}")

**Lectura:**
- Hay que correr el cuantil **4 días** hacia abajo. Con eso cubre el 90 % donde importa.
- El costo de la calibración: el RUL pesimista baja, y el plan cambiará las multiplicadoras antes. Ese balance lo decide el MILP.

## 4. La flota de hoy: del modelo al plan
Entrenamos con las 100 turbinas de la historia y predecimos la **flota en operación**: 40 turbinas que siguen funcionando.
De cada una vemos el último día: sus sensores de hoy. Como es una simulación, también sabemos su **RUL real**. Lo guardamos aparte para evaluar al final, como si fuera el futuro.

In [ ]:
modelo = gb().fit(X, y)
modelo_q10 = gb(loss="quantile", quantile=0.10).fit(X, y)

H, CUADRILLAS = 60, 1


def flota_hoy(n, semilla):
    hoy = simular(n, semilla, censurar=True).groupby("turbina").tail(1).set_index("turbina")
    hoy["rul_pred"] = modelo.predict(hoy[X_COLS]).round(1)
    hoy["rul_q10"] = np.maximum(1, modelo_q10.predict(hoy[X_COLS]) - delta).round(1)
    return hoy


flota = flota_hoy(40, semilla=7)
print("Turbinas:", len(flota), "| con RUL predicho <= 60 días:", (flota["rul_pred"] <= H).sum(), "| con RUL real <= 60:", (flota["rul"] <= H).sum())
flota[["dia", "rul_pred", "rul_q10", "rul"]].sort_values("rul_pred").head(8)

Cuatro formas de pasar la predicción al MILP. Cambia **solo** la tabla de costos; el modelo es el mismo.

| Plan | RUL que ve el MILP | Qué supone |
|---|---|---|
| `punto` | $\hat{r}_e$ | el pronóstico es exacto |
| `cuantil` | $\hat{q}^{\,\text{cal}}_e$ | el pronóstico pesimista es exacto |
| `escenarios` | $\hat{r}_e$ + los errores de su tramo | el error de mañana se parece al de la validación |
| `perfecto` | el RUL real | sabes el futuro (solo para comparar: el EVPI de la lección 8) |

In [ ]:
TRAMOS = errores["tramo"].cat.categories
ERRORES_TRAMO = {t: errores.loc[errores["tramo"] == t, "error"].sample(500, random_state=0).to_numpy() for t in TRAMOS}


def escenarios_rul(pred):
    # pred + los errores de validación de su tramo (500 escenarios por turbina)
    t = pd.cut([min(pred, 150)], [-1, 20, 40, 60, 90, 151], labels=TRAMOS)[0]
    return np.maximum(1, pred + ERRORES_TRAMO[t])


def supuestos(hoy):
    return {"punto": {e: [r] for e, r in hoy["rul_pred"].items()},
            "cuantil": {e: [r] for e, r in hoy["rul_q10"].items()},
            "escenarios": {e: escenarios_rul(r) for e, r in hoy["rul_pred"].items()},
            "perfecto": {e: [r] for e, r in hoy["rul"].items()}}


print("Escenarios de una turbina con RUL predicho de 30 días (primeros 10):", escenarios_rul(30)[:10].round(0))

In [ ]:
def evaluar(hoy, cuadrillas=CUADRILLAS):
    filas, planes = [], {}
    for nombre, rul_supuesto in supuestos(hoy).items():
        m = plan_mantenimiento(tabla_costos(rul_supuesto, H), cuadrillas)
        dia = pd.Series(m.dia)
        real = np.array([costo(dia[e], hoy.loc[e, "rul"], H).item() for e in hoy.index])  # el costo con el RUL que de verdad tenía
        fallas = (real == FALLA).sum()
        filas.append({"plan": nombre, "cambios": int((dia > 0).sum()), "fallas": int(fallas),
                      "dias_desperdiciados": int(real[real < FALLA].sum() / VALOR_DIA), "costo_real": real.sum(),
                      "costo_que_cree": pyo.value(m.obj), "segundos": round(m.segundos, 2)})
        planes[nombre] = dia
    return pd.DataFrame(filas), pd.DataFrame(planes)


resultado, planes = evaluar(flota)
resultado.assign(costo_real=resultado["costo_real"].map(eur), costo_que_cree=resultado["costo_que_cree"].map(eur))

**Lectura** (los planes pueden cambiar en detalle con la versión de HiGHS si hay empates; los totales, poco):
- El plan con el **punto** cree que cuesta 10.500 € y cuesta **472.000 €**, con **4 fallas**. Programa cada cambio al filo del RUL predicho: cuando el modelo se pasa, llega tarde.
- El plan con el **cuantil** es el peor: **960.000 €** y 5 fallas. Dos razones:
  - cambia muchas multiplicadoras con semanas de anticipación (383 días desperdiciados);
  - 4 turbinas tienen cuantil de 1 día y la cuadrilla solo hace una por día. Para el MILP, las otras 3 ya están perdidas: cambiarlas el día 2 o no cambiarlas
    "cuesta" lo mismo, 60.000 €. **Las abandona**, aunque su RUL real era de 12, 25 y 26 días. Con un solo número, el riesgo es todo o nada.
- El plan con **escenarios** es el más barato: **412.000 €**, 2 fallas. Para cada turbina y día sabe la **probabilidad** de llegar tarde, y la compara con los días que desperdicia.
- El plan **perfecto** marca el piso: 3.000 €. La distancia entre `escenarios` y `perfecto` es lo que vale **mejorar el modelo de ML** (el EVPI de la lección 8).

### 4.1 La solución, de vuelta a tablas

In [ ]:
plan_hoy = flota[["rul_pred", "rul_q10", "rul"]].join(planes.add_prefix("dia_"))
sin_cambio = plan_hoy["dia_escenarios"] == 0
plan_hoy["resultado_escenarios"] = np.where(sin_cambio & (plan_hoy["rul"] > H), "queda para el próximo plan",
                                   np.where(sin_cambio, "FALLA sin cambio",
                                   np.where(plan_hoy["dia_escenarios"] <= plan_hoy["rul"], "a tiempo", "FALLA: llegó tarde")))
# Top 5: las turbinas más urgentes según el modelo
plan_hoy.nsmallest(5, "rul_pred")

In [ ]:
# Las turbinas donde falló el plan con el punto
plan_hoy.assign(falla_punto=(plan_hoy["dia_punto"] > plan_hoy["rul"]) | ((plan_hoy["dia_punto"] == 0) & (plan_hoy["rul"] <= H))).query("falla_punto")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
orden = plan_hoy.query("rul_pred <= 90").sort_values("rul_pred")
pos = np.arange(len(orden))
ax.scatter(pos, orden["rul"], marker="x", color="black", label="RUL real", zorder=3)
ax.scatter(pos, orden["rul_pred"], color="#94a3b8", label="RUL predicho")
for nombre, color, desplaza in [("punto", "#f97316", -0.15), ("escenarios", "#2563eb", 0.15)]:
    d = orden[f"dia_{nombre}"]
    usa = (d > 0).to_numpy()
    ax.scatter(pos[usa] + desplaza, d[usa], color=color, marker="s", s=25, label=f"día del cambio: {nombre}")
ax.axhline(H, color="gray", ls=":", lw=1)
ax.set_xticks(pos, orden.index, rotation=90, fontsize=7)
ax.set(ylabel="días", title="RUL predicho <= 90 días. Cuadrado sobre la x: el cambio llegó tarde")
ax.legend(fontsize=7, ncol=4, loc="upper left")
plt.tight_layout()
plt.show()

**Lectura para el jefe de mantenimiento:**
- Con el plan por escenarios, los próximos 60 días cuestan **412.000 €** en esta flota de 40 turbinas, contra **472.000 €** si se programa al filo del pronóstico. Con el RUL pesimista, 960.000 €.
- La diferencia sale de **fallas evitadas**: 2 contra 4. Cada una son 60.000 € y semanas de turbina parada.
- Las 2 fallas que quedan: T014, que el modelo veía lejos (83 días) y falló a los 37; y T034, cambiada el día 26 cuando le quedaban 24. Ningún plan con este modelo las salva a las dos.
- El plan no es "cambiar todo pronto": las turbinas con RUL predicho cerca del horizonte quedan para el próximo plan, que se rehace cada semana con datos nuevos.
- Un modelo de ML mejor vale como mucho la distancia al plan perfecto. Ese número dice cuánto invertir en sensores o en datos.

## 5. El modelo crece: de 40 a 640 turbinas
Un operador con varios parques: más turbinas y más cuadrillas (una cuadrilla por cada 40 turbinas). Para cada tamaño, 3 flotas distintas.
Medimos el tamaño del MILP, el tiempo y el costo real de cada plan **por turbina**.
Esta celda tarda alrededor de 30 segundos.

In [ ]:
filas = []
for n, cuadrillas in [(40, 1), (160, 4), (640, 16)]:
    for semilla in [101, 102, 103]:
        hoy = flota_hoy(n, semilla)
        res, _ = evaluar(hoy, cuadrillas)
        for r in res.itertuples():
            filas.append({"turbinas": n, "cuadrillas": cuadrillas, "flota": semilla, "plan": r.plan, "fallas": r.fallas,
                          "costo_por_turbina": r.costo_real / n, "segundos": r.segundos})
escala = pd.DataFrame(filas)
resumen = escala.pivot_table(index=["turbinas", "cuadrillas"], columns="plan", values="costo_por_turbina", aggfunc="mean").round(0)
resumen

In [ ]:
escala.pivot_table(index="turbinas", columns="plan", values=["fallas", "segundos"], aggfunc="mean").round(2)

In [ ]:
grande = flota_hoy(640, 103)
mg = plan_mantenimiento(tabla_costos(supuestos(grande)["escenarios"], H), cuadrillas=16)
print("Tamaño con 640 turbinas:", tamano(mg), f"| {mg.segundos:.1f} s")

**Cómo se lee:**
- En los 3 tamaños, `escenarios` es el más barato: entre **13 % y 21 % menos** por turbina que el `punto` (con 640 turbinas, 13.586 € contra 15.861 €).
- El `punto` tiene unas **2 veces más fallas** que `escenarios` (135 contra 63 con 640 turbinas).
- El `cuantil` tiene **menos fallas** que `escenarios` (42 contra 63), pero desperdicia tanta vida que cuesta más. Menos fallas no es lo mismo que menos costo.
- Con 640 turbinas, el MILP tiene 39.040 binarias y resuelve en **menos de 1 s**: es una asignación (matriz TU, lección 4). Lo que más tarda es armar la tabla de costos y el modelo en Pyomo.

**Los dos tamaños.** Modelo: $\lvert E \rvert (H + 1)$ binarias, lineal. Espacio de búsqueda: $(H + 1)^{\lvert E \rvert}$; con 640 turbinas, $61^{640} \approx 10^{1143}$.

## 6. La métrica que importa: el costo de la decisión
Juntemos en una tabla lo que dice cada métrica sobre los planes de la flota de 40.

In [ ]:
mae_supuesto = {"punto": mean_absolute_error(flota["rul"].clip(upper=150), flota["rul_pred"]),
                "cuantil": mean_absolute_error(flota["rul"].clip(upper=150), flota["rul_q10"]),
                "escenarios": mean_absolute_error(flota["rul"].clip(upper=150), flota["rul_pred"]), "perfecto": 0.0}
resultado.assign(MAE_del_RUL=resultado["plan"].map(mae_supuesto).round(1), costo_real=resultado["costo_real"].map(eur))[
    ["plan", "MAE_del_RUL", "fallas", "dias_desperdiciados", "costo_real"]]

**Lectura:**
- `punto` y `escenarios` usan **la misma predicción**: el mismo MAE. Sus costos son muy distintos. El MAE no ve cómo se usa el error.
- El `cuantil` tiene el doble de MAE (está corrido a propósito). En la sección 5 da menos fallas que los otros dos; aquí no, porque abandona las turbinas que cree perdidas.
- **Regla:** el modelo de ML se elige y se ajusta mirando el **costo de la decisión** en validación, no solo el MAE.
  Si el error caro es llegar tarde, se busca un modelo que casi nunca se pase, aunque su MAE suba.
  Llevado al extremo, esto es ***decision-focused learning*** (o *smart predict-then-optimize*): entrenar el modelo con la pérdida del optimizador.

## 7. ¿Cómo conectar el ML con el MILP?
Una sola regla para empezar:

> **Pasa al optimizador la incertidumbre, no solo el punto. Si la segunda etapa no decide nada, precalcula el costo esperado; si decide, usa escenarios (lección 8).**

| Forma | Qué recibe el MILP | Cuándo |
|---|---|---|
| Predict-then-optimize con el punto | $\hat{y}$ | errores baratos y simétricos, o como primera versión |
| Cuantil o margen de seguridad | $\hat{q}_\tau$ calibrado | una restricción dura ("no pasar del RUL") y poco tiempo |
| Escenarios desde los errores | $\hat{y}$ + errores de validación | costos asimétricos; hay datos para medir el error |
| Optimización robusta | un intervalo $[\hat{y} - \delta, \hat{y} + \delta]$ | sin datos de error; se protege del peor caso |
| Decision-focused learning | el modelo se entrena con el costo del MILP | cuando lo demás no basta; más complejo de mantener |

**En producción** (lo verás el domingo): el pipeline corre cada semana. Datos nuevos → features → predicción → escenarios → MILP → plan.
Se monitorea la **cobertura** del cuantil y el MAE por tramo: si el modelo se degrada, el plan lo sufre antes de que alguien lo note.

## 8. Resumen: conceptos de hoy
| Concepto | Qué es | En la lección | En inglés |
|---|---|---|---|
| Vida útil remanente | Días hasta la falla | RUL de las multiplicadoras | remaining useful life (RUL) |
| Predecir y luego optimizar | El ML predice, el MILP decide | RUL → plan de 60 días | predict-then-optimize |
| Fuga de datos | Validar con información que en la realidad no tendrías | 10 días por fila contra 15 por turbina | data leakage |
| Validación por grupos | Cada turbina entera en train o en test | `GroupKFold` | group k-fold cross-validation |
| Línea base | Un modelo simple para comparar | promedio: 44 días; lineal: 19 | baseline |
| Gradient boosting | Suma de árboles, cada uno corrige al anterior | 15 días | gradient boosting |
| Error por tramo | El error depende de la zona de la predicción | 7 días cerca, 19 lejos | heteroscedastic error |
| Cuantil | Un valor que el real supera con probabilidad $1 - \tau$ | RUL pesimista | quantile regression |
| Pérdida pinball | Castiga distinto pasarse y quedarse corto | `loss="quantile"` | pinball loss |
| Cobertura | Qué tan seguido el real supera al cuantil | 81 % en vez de 90 % | coverage |
| Calibración conformal | Correr el cuantil con errores fuera de la muestra | δ = 4 días | conformal prediction |
| Costo esperado precalculado | El promedio sobre escenarios entra como dato | $\bar{c}_{ed}$ | expected cost coefficients |
| Costo de la decisión | Evaluar el modelo por lo que cuesta el plan | mismo MAE, costos distintos | decision-focused evaluation |

### Notación de hoy
| Símbolo | Significa | En la lección |
|---|---|---|
| $\hat{y}$, $\hat{r}_e$ | valor predicho | RUL predicho |
| $\text{MAE} = \frac{1}{n} \sum_i \lvert y_i - \hat{y}_i \rvert$ | error absoluto medio | 15 días |
| $\lvert \cdot \rvert$ | valor absoluto (y también tamaño de un conjunto, $\lvert E \rvert$) | errores; número de turbinas |
| $\tau$, $\rho_\tau$ | cuantil y pérdida pinball | $\tau = 0{,}1$ |
| $\hat{q}^{\,\text{cal}} = \hat{q} - \delta$ | cuantil calibrado | δ = 4 días |
| $r_{es}$, $S_e$ | RUL de la turbina $e$ en el escenario $s$ | 500 escenarios por turbina |
| $c(d, r)$, $\bar{c}_{ed}$ | costo de un cambio y su promedio sobre escenarios | la tabla de costos |
| $D \setminus \{0\}$ | días de trabajo | cuadrillas |
| $(H + 1)^{\lvert E \rvert}$ | espacio de búsqueda del plan | $61^{40} \approx 10^{71}$ |

## 9. Ejercicios
1. **Una falla más cara.** La falla sube de 60.000 € a 200.000 € (una turbina offshore parada en invierno). Antes de resolver, adivina: ¿el plan por escenarios cambia más o menos multiplicadoras? ¿Antes o después?
   Cambia `FALLA` y corre `evaluar(flota)`. Al final, vuelve a dejarla en 60.000.
2. **¿Contratar otra cuadrilla?** Una segunda cuadrilla cuesta 40.000 € por los 60 días. Con el plan por escenarios en la flota de 40, ¿vale la pena? Usa `evaluar(flota, cuadrillas=2)`.
3. **¿Qué sensor importa?** Entrena con 80 turbinas y mide con 20 (`GroupShuffleSplit`). Calcula `permutation_importance` en las 20.
   ¿Qué feature pesa más? ¿El nivel o el cambio en 10 días?

In [ ]:
# Escribe aquí tus intentos

## En la entrevista
Pregunta típica: *"Tell me about a project where you combined machine learning and optimization."*
> *"Predictive maintenance: a gradient boosting model predicts the remaining useful life of each gearbox from sensor features, and a MILP schedules
> maintenance with crew capacity. The key points are three. First, validation: I use group k-fold by asset, because row-wise splits leak;
> in my example row-wise CV reported 10 days of error and the honest number was 15. Second, uncertainty: the error is not uniform, it grows with the horizon,
> so I don't pass just the point forecast. I build scenarios from out-of-fold errors per prediction range and precompute expected costs for each asset and day.
> Third, I evaluate the model by the cost of the decision, not only by MAE: with the same predictions, the point-forecast plan had several failures,
> the scenario-based plan was 13 to 21% cheaper per turbine."*

Seguimiento: *"Why not just use a quantile?"*
> *"A quantile is a good first step, but it needs calibration: my 10% quantile only covered 81% near failure, so I shifted it with conformal calibration.
> Even calibrated, it's one number per asset, so the optimizer treats it as certain: it maintains too early, and when crew capacity is tight it gives up
> on assets it believes are lost, because any late day costs the same. Scenarios give it the probability of being late for each asset and day,
> so it trades wasted life against failure risk. In my example that was the cheapest plan at every fleet size."*

Seguimiento: *"How would you deploy this?"*
> *"A weekly batch pipeline: ingest sensors, compute features, predict, build the cost table, solve, publish the plan. I'd monitor data drift,
> the coverage of the quantile and the realized cost of the plan, and retrain when coverage drops."*

## Soluciones (no mires antes de intentar)

In [ ]:
# Ejercicio 1
FALLA = 200_000
caro, planes_caro = evaluar(flota)
FALLA = 60_000
comparar = pd.DataFrame({"día con falla 60.000": planes["escenarios"], "día con falla 200.000": planes_caro["escenarios"]})
print(caro[["plan", "cambios", "fallas", "dias_desperdiciados"]].to_string(index=False))
comparar[comparar.iloc[:, 0] != comparar.iloc[:, 1]]

Con la falla a 200.000 €, el plan por escenarios cambia **19** multiplicadoras en vez de 11, y **antes**: T031 pasa del día 24 al 6, T034 del 26 al 7.
También entran turbinas que antes quedaban para el próximo plan (T000, T005, T011…): con la falla tan cara, un riesgo chico de fallar dentro de los 60 días ya pesa más que el desperdicio.
Resultado: **1 falla** en vez de 2, pero 839 días desperdiciados en vez de 212. El plan con el punto **no cambia nada**: con el pronóstico exacto, nunca cree que va a fallar, así que el precio de la falla no le importa.
Ese es el problema del punto: el optimizador no puede reaccionar a un riesgo que no ve.

In [ ]:
# Ejercicio 2
dos, _ = evaluar(flota, cuadrillas=2)
uno_e, dos_e = resultado.set_index("plan").loc["escenarios"], dos.set_index("plan").loc["escenarios"]
print("Costo esperado (lo que cree el plan):", eur(uno_e["costo_que_cree"]), "con 1 |", eur(dos_e["costo_que_cree"]), "con 2 | ahorro", eur(uno_e["costo_que_cree"] - dos_e["costo_que_cree"]))
print("Costo real en esta flota:            ", eur(uno_e["costo_real"]), "con 1 |", eur(dos_e["costo_real"]), "con 2 | ahorro", eur(uno_e["costo_real"] - dos_e["costo_real"]))
# Lo mismo en las 3 flotas de 160 turbinas: 4 cuadrillas contra 8
for semilla in [101, 102, 103]:
    hoy = flota_hoy(160, semilla)
    a = evaluar(hoy, 4)[0].set_index("plan").loc["escenarios"]
    b = evaluar(hoy, 8)[0].set_index("plan").loc["escenarios"]
    print(f"160 turbinas, flota {semilla}: ahorro esperado con 4 cuadrillas más {eur(a['costo_que_cree'] - b['costo_que_cree'])} | real {eur(a['costo_real'] - b['costo_real'])}")

Con una cuadrilla más, el plan espera ahorrar unos **7.000 €** en 60 días. La cuadrilla cuesta 40.000 €: **no vale la pena**.
En las flotas de 160 turbinas pasa lo mismo: 4 cuadrillas más ahorran de 22.000 a 37.000 € esperados, unos 6.000 a 9.000 € por cuadrilla.
Fíjate en la columna "real": en esta flota, con 2 cuadrillas el costo real **sube** 3.000 €. Es ruido: el plan es óptimo en promedio, no en cada flota.
**Regla:** las decisiones de capacidad se toman con el **costo esperado** (o con muchas flotas simuladas), no con lo que pasó en una.

In [ ]:
# Ejercicio 3
from sklearn.inspection import permutation_importance
from sklearn.model_selection import GroupShuffleSplit

train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X, y, grupos))
mod = gb().fit(X.iloc[train], y.iloc[train])
imp = permutation_importance(mod, X.iloc[test], y.iloc[test], scoring="neg_mean_absolute_error", n_repeats=5, random_state=0)
pd.Series(imp.importances_mean, index=X_COLS).sort_values(ascending=False).round(1).rename("sube_el_MAE_en_dias").head(6)

La feature que más pesa es **`vibracion_m10`**: si se desordena, el MAE sube unos 36 días. Le sigue la edad (`dia`), con 4 días. Las demás casi no aportan.
Pesa el **nivel** promedio, no el cambio en 10 días: en estos datos la vibración crece mucho más que su nivel base, así que el nivel ya dice cuánto daño hay.
Importa medir la importancia en turbinas que el modelo **no vio** (las 20 de test): en train, el modelo también "usa" las features que le sirven para memorizar.